# C03. Every thread gets a struct

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tamnd/cpython-internals/blob/main/lessons/c03-every-thread-gets-a-struct/c03.ipynb)

C01 measured the lock and C02 measured what it was keeping safe. Both lessons kept saying things like "a thread hands the lock over" without ever saying what a thread is to the interpreter.

It is a struct, and you can go and look at it. This lesson walks the interpreter's own list of threads from Python, using about fifteen lines of `ctypes` and nothing else.

![the operating system's idea of a thread on one side and the interpreter's on the other](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/c03-every-thread-gets-a-struct/diagrams/what-a-thread-is-here.svg)

## About the source references

Now and then this lesson points at CPython's own source, like this: `Include/cpython/pystate.h:66-101@v3.15.0rc1`.

Read it as three parts: the file, the lines, and the release those line numbers belong to. Sometimes there is a fourth part after a `#`, which is the name of the thing those lines are inside.

Every reference is a link, and every one is checked against the pinned source on each change, so a stale reference fails the build instead of sending you somewhere wrong. You never have to read any of it. The references are there so you can go deeper when you want to, and so you can check that this lesson is not making things up.

## Setup

Colab does not come with the small package these lessons use, so the next cell installs it. If you are running this from a checkout of the repository it is already installed and the cell does nothing.

In [ ]:
import sys

if sys.version_info < (3, 14):
    print("This lesson needs CPython 3.14 or newer.")
    print(f"This runtime is {sys.version.split()[0]}, and the cells below will not run on it.")
else:
    try:
        import pyxray
    except ImportError:
        %pip install -q "pyxray @ git+https://github.com/tamnd/cpython-internals@main#subdirectory=pyxray"
        import pyxray

## Which Python is this

Most of this lesson runs anywhere. Two things do not: a browser tab cannot start a thread, and it cannot start another process either. Those cells check first and say so rather than printing nonsense. In Colab or from a checkout, everything runs.

## Which interpreter is this

In [ ]:
import pyxray

pyxray.show()

## One struct per thread

Start with the thing itself. Every thread running Python has a [thread state](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#thread-state), which in C is a `PyThreadState` [Include/cpython/pystate.h:66-101@v3.15.0rc1#_ts](https://github.com/python/cpython/blob/v3.15.0rc1/Include/cpython/pystate.h#L66-L101). It is a plain struct with about forty fields in it, and the first three are the ones this section is about: a `prev` pointer, a `next` pointer, and the interpreter it belongs to.

![nine fields of one thread state, from the list pointers at the bottom to the state field at the top](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/c03-every-thread-gets-a-struct/diagrams/what-rides-on-the-state.svg)

The `prev` and `next` are there because the interpreter keeps every thread state on one linked list. The head of it lives on the interpreter, next to a counter used to hand out ids [Include/internal/pycore_interp_structs.h:867-881@v3.15.0rc1#pythreads](https://github.com/python/cpython/blob/v3.15.0rc1/Include/internal/pycore_interp_structs.h#L867-L881). Making a thread state bumps that counter, takes a lock, and pushes the new state onto the front [Python/pystate.c:1632-1671@v3.15.0rc1#new_threadstate](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pystate.c#L1632-L1671), so the list runs newest to oldest and the last thing on it is always the main thread [Python/pystate.c:1618-1630@v3.15.0rc1#add_threadstate](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pystate.c#L1618-L1630).

![the interpreter's head pointer and four thread states chained by their next pointers](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/c03-every-thread-gets-a-struct/diagrams/the-list-newest-first.svg)

None of that is hidden. Four of the functions that walk it are public C API, which means `ctypes` can call them [Python/pystate.c:2671-2679@v3.15.0rc1#PyInterpreterState_ThreadHead](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pystate.c#L2671-L2679). The cell below asks for the current thread's state, asks which interpreter it belongs to, and then follows the chain, printing the id of every state it finds [Python/pystate.c:2135-2140@v3.15.0rc1#PyThreadState_GetID](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pystate.c#L2135-L2140).

Watch the list grow when three threads start and shrink back when they finish.

Every thread has its own PyThreadState, the interpreter keeps them all on one linked list with the newest at the front, and a state leaves the list when its thread finishes

In [ ]:
import ctypes
import threading

api = ctypes.pythonapi
NO_THREADS = "  this build cannot start a thread, so the list will never have more than one"


def ask(name, args, result):
    """Point ctypes at one C API function. Every one used here is public and documented."""
    fn = getattr(api, name)
    fn.argtypes = args
    fn.restype = result
    return fn


current_state = ask("PyThreadState_Get", [], ctypes.c_void_p)
state_id = ask("PyThreadState_GetID", [ctypes.c_void_p], ctypes.c_uint64)
interpreter_of = ask("PyThreadState_GetInterpreter", [ctypes.c_void_p], ctypes.c_void_p)
first_state = ask("PyInterpreterState_ThreadHead", [ctypes.c_void_p], ctypes.c_void_p)
next_state = ask("PyThreadState_Next", [ctypes.c_void_p], ctypes.c_void_p)

INTERP = interpreter_of(current_state())


def walk():
    """Follow interp->threads.head and then ->next, the way the interpreter walks it."""
    found = []
    state = first_state(INTERP)
    while state:
        found.append(state_id(state))
        state = next_state(state)
    return found


def threads_work():
    """Some runtimes cannot start a thread at all. A browser tab is one of them."""
    try:
        probe = threading.Thread(target=lambda: None)
        probe.start()
        probe.join()
    except RuntimeError:
        return False
    return True


THREADS_WORK = threads_work()


def park(count, body=None):
    """Start count threads and hold every one of them still until release() lets them go."""
    ready = threading.Semaphore(0)
    go = threading.Event()

    def pause():
        ready.release()
        go.wait()

    def wrapper():
        if body is None:
            pause()
        else:
            body(pause)

    hands = [threading.Thread(target=wrapper) for _ in range(count)]
    for hand in hands:
        hand.start()
    for _ in hands:
        ready.acquire()
    return hands, go


def release(hands, go):
    go.set()
    for hand in hands:
        hand.join()


here = current_state()
print("  this thread's state sits at", hex(here), "and its id is", state_id(here))
print("  the interpreter it belongs to sits at", hex(INTERP))
print("  ids on the list right now:", walk())
if not THREADS_WORK:
    print(NO_THREADS)
else:
    crew = park(3)
    print("  with three more threads parked:", walk())
    release(*crew)
    print("  once those three have finished:", walk())

> **Version note.** the two addresses are wherever this process happened to put them, and the ids depend on how many threads the runtime has already started, which is one in a plain script and several in a notebook kernel

## Three numbers, and only one of them is yours to trust

There are three ways to ask which thread you are, and they are not interchangeable.

The first is the state id you just printed. The interpreter hands those out itself, one higher each time, under the same lock that pushes the state onto the list. It never reuses one.

The other two come from the operating system, and they get filled in later. A thread state is created by whoever asked for the thread, but `thread_id` and `native_thread_id` are written by the new thread itself, the first time it binds to its state [Python/pystate.c:162-192@v3.15.0rc1#bind_tstate](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pystate.c#L162-L192). They are the two fields sitting next to the scratch dict in the header [Include/cpython/pystate.h:161-172@v3.15.0rc1#native_thread_id](https://github.com/python/cpython/blob/v3.15.0rc1/Include/cpython/pystate.h#L161-L172).

`threading.get_ident()` returns the first of those, and the documentation is honest about it: the value may be recycled when a thread exits. The cell below is that sentence made uncomfortable. Three threads, joined, three more, joined, three more. The state ids climb. The idents come back around.

![the three identifiers, who writes each one, when, and whether it ever gets handed out twice](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/c03-every-thread-gets-a-struct/diagrams/three-numbers-one-thread.svg)

If you have ever keyed a dictionary on `threading.get_ident()` and kept it after the thread finished, this is the bug.

A thread state id is never handed out twice, while threading.get_ident() is, so the same ident can name two different threads over the life of one program

In [ ]:
def one_wave(size):
    """Start size threads at once, and have each of them report the three numbers."""
    seen = []
    guard = threading.Lock()

    def report(pause):
        state = current_state()
        with guard:
            seen.append((state_id(state), threading.get_ident(), threading.get_native_id()))
        pause()

    crew = park(size, report)
    release(*crew)
    return sorted(seen)


if not THREADS_WORK:
    print(NO_THREADS)
else:
    print(f"  {'wave':<6}{'state id':>10}{'get_ident()':>16}{'get_native_id()':>18}")
    for wave in (1, 2, 3):
        for state, ident, native in one_wave(3):
            print(f"  {wave:<6}{state:>10}{ident:>16}{native:>18}")

> **Version note.** the ids depend on how many threads this runtime has already made, and the idents and native ids are whatever the operating system handed out, but the state ids climb and the idents repeat from one wave to the next on any build

## sys._current_frames is that same walk

`sys._current_frames()` gives you a dict from thread to frame, and it is worth knowing where that dict comes from. It stops every thread, takes the lock on the list, walks it, and for each state reads `current_frame` and uses `thread_id` as the key [Python/pystate.c:2691-2730@v3.15.0rc1#_PyThread_CurrentFrames](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pystate.c#L2691-L2730).

So it is the walk from the first cell, with two fields read instead of one. That also explains the keys: they are `thread_id`, the recycled one, which is why the dict matches `threading.get_ident()` and not the state ids.

The cell parks three threads and prints all three views at once.

The keys of sys._current_frames() are exactly the idents of the live threads, because the dict is built by walking the same list of thread states

In [ ]:
def three_views():
    """The same set of threads, seen through the C list, through sys, and through threading."""
    print("  ids on the interpreter's list:", sorted(walk()))
    frames = sorted(sys._current_frames())
    idents = sorted(hand.ident for hand in threading.enumerate())
    print("  keys in sys._current_frames():", frames)
    print("  idents from threading:       ", idents)
    print("  the last two are the same set:", frames == idents)


if not THREADS_WORK:
    three_views()
else:
    crew = park(3)
    three_views()
    release(*crew)

> **Version note.** the ids and idents are whatever this run produced, and the count depends on whether anything else in the runtime is holding threads open, but the last line is True everywhere

## What only this thread can see

Once you know a thread has its own struct, a lot of Python stops being magic.

`threading.local` is the obvious one. Each thread state holds a small key object, and every `threading.local` is a dict from those keys to that thread's values [Modules/_threadmodule.c:1376-1388@v3.15.0rc1#threading_local_key](https://github.com/python/cpython/blob/v3.15.0rc1/Modules/_threadmodule.c#L1376-L1388). The key is made on first use and lives on the state [Modules/_threadmodule.c:1600-1623@v3.15.0rc1#create_localdummies](https://github.com/python/cpython/blob/v3.15.0rc1/Modules/_threadmodule.c#L1600-L1623), so when the thread goes, its values go with it.

There is a second, older per thread dict, and `PyThreadState_GetDict()` hands it straight to you [Python/pystate.c:2101-2108@v3.15.0rc1#PyThreadState_GetDict](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pystate.c#L2101-L2108). The interpreter uses it for its own bookkeeping, and there is one use of it you have definitely seen. When you print a list that contains itself, the `...` comes from a list of objects currently being printed, kept in this dict, on this thread [Objects/object.c:3106-3137@v3.15.0rc1#Py_ReprEnter](https://github.com/python/cpython/blob/v3.15.0rc1/Objects/object.c#L3106-L3137). That is why two threads can print two self referential structures at once without confusing each other.

The exception being handled is on the state too, which is what `sys._current_exceptions()` reads, using the same walk as before. The cell parks a worker inside an `except` block and asks from the main thread.

The threading.local values, the scratch dict and the exception being handled all live on the thread state, so one thread never sees another thread's

In [ ]:
state_dict = ask("PyThreadState_GetDict", [], ctypes.c_void_p)


def scratch():
    """The dict PyThreadState_GetDict() hands back, which belongs to this thread alone."""
    return ctypes.cast(state_dict(), ctypes.py_object).value


local = threading.local()
local.note = "written by the main thread"

loop = []
loop.append(loop)
print("  a list holding itself prints as", repr(loop))
print("  and now this thread's scratch dict holds", sorted(scratch()))

answers = {}


def look(pause):
    local.note = "written by the worker"
    scratch()["mine"] = "and this key is the worker's"
    answers["note"] = local.note
    answers["keys"] = sorted(scratch())
    try:
        raise ValueError("held open inside the worker")
    except ValueError:
        pause()


if not THREADS_WORK:
    print(NO_THREADS)
else:
    crew = park(1, look)
    print("  the worker's local.note:  ", answers["note"])
    print("  the main thread's is still:", local.note)
    print("  the worker's scratch dict:", answers["keys"])
    print("  the main thread's:        ", sorted(scratch()))
    for ident, held in sorted(sys._current_exceptions().items()):
        print(f"  thread {ident} is handling {held!r}")
    release(*crew)

> **Version note.** the idents are whatever the operating system handed out, and how many threads show up in the last two lines depends on what else the runtime is keeping alive, but only the worker is ever holding an exception

## Attaching and detaching

Now the field the whole of C01 was really about.

A thread state has an `int` called `state`, and it holds one of four values [Include/internal/pycore_pystate.h:20-49@v3.15.0rc1#_Py_THREAD_SUSPENDED](https://github.com/python/cpython/blob/v3.15.0rc1/Include/internal/pycore_pystate.h#L20-L49). Attached means the thread is running Python. Detached means it is not, either because it is sitting in a C function or because it is waiting for something. The other two are done to a thread rather than by it.

![the four values the state field can hold, who is allowed to write each one, and what the thread is doing in it](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/c03-every-thread-gets-a-struct/diagrams/attached-detached-suspended.svg)

[attach and detach](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#attach-and-detach) is the whole handshake. Attaching takes the lock, sets the field, and marks the state active [Python/pystate.c:2225-2251@v3.15.0rc1#_PyThreadState_Attach](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pystate.c#L2225-L2251). Detaching does the reverse and releases the lock last [Python/pystate.c:2284-2300@v3.15.0rc1#detach_thread](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pystate.c#L2284-L2300). On a build with the [GIL](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#gil) the setting of the field is a plain store, because only one thread can be doing it. On the [free threaded build](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#free-threaded-build) it is a compare and exchange, because several can [Python/pystate.c:2178-2191@v3.15.0rc1#tstate_try_attach](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pystate.c#L2178-L2191).

And `Py_BEGIN_ALLOW_THREADS`, the macro C01 spent a section on, is exactly this. It expands to `PyEval_SaveThread`, which is one line: detach [Python/ceval_gil.c:642-663@v3.15.0rc1#PyEval_SaveThread](https://github.com/python/cpython/blob/v3.15.0rc1/Python/ceval_gil.c#L642-L663). The matching `PyEval_RestoreThread` attaches again. "Release the GIL" and "detach the thread state" are the same sentence said two ways, and the second one is the one that is still true when there is no GIL.

There is a nice detail hiding in the attach path. C02 showed that a [critical section](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#critical-section) is allowed to let go of its lock partway through. That is where it happens: detaching suspends whatever critical sections the thread was holding, and attaching picks them back up [Python/pystate.c:2302-2306@v3.15.0rc1#_PyThreadState_Detach](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pystate.c#L2302-L2306). The stack of them is one more field on the thread state.

The third value, suspended, is what [stop the world](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#stop-the-world) does. A thread asked to suspend does not stop where it is. It carries on to its next [periodic check](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#periodic-check), and only then parks itself and waits [Python/pystate.c:2204-2223@v3.15.0rc1#tstate_wait_attach](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pystate.c#L2204-L2223). The collector lessons in M7 leaned on this without saying where it lived. It lives here, in one int.

Py_BEGIN_ALLOW_THREADS is a detach of the thread state and Py_END_ALLOW_THREADS is an attach, which is why releasing the GIL and detaching are the same operation

## The thread that never attaches again

The fourth value is the one with consequences you have probably hit.

When the interpreter shuts down it does not ask the other threads to stop. It writes a 3 into the `state` field of each of them and moves on [Python/pystate.c:2341-2348@v3.15.0rc1#_PyThreadState_SetShuttingDown](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pystate.c#L2341-L2348). A [daemon thread](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#daemon-thread) that is in the middle of a loop keeps going, because nothing has interrupted it yet. Then it reaches its next periodic check, tries to attach, reads the 3, and is hung where it stands [Python/pystate.c:3199-3212@v3.15.0rc1#_PyThreadState_HangThread](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pystate.c#L3199-L3212).

Hung, not stopped. No exception is raised in it, so no `finally` runs, no `with` block exits, and nothing it was holding is released [Python/pystate.c:3191-3204@v3.15.0rc1#_PyThreadState_MustExit](https://github.com/python/cpython/blob/v3.15.0rc1/Python/pystate.c#L3191-L3204).

![the main thread returning, shutdown writing to every other state, and the daemon getting hung at its next check](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/c03-every-thread-gets-a-struct/diagrams/the-daemon-at-shutdown.svg)

The cell starts a child interpreter with a daemon thread counting in a loop, lets the main thread sleep briefly and return, and reads back what the child managed to print. The count it reaches is different every run. Whether the `finally` block ran is not.

A daemon thread still running at shutdown is hung at its next periodic check, and its finally blocks do not run

In [ ]:
import subprocess

CHILD = """
import threading
import time


def body():
    n = 0
    try:
        while True:
            n += 1
            if n % 1000000 == 0:
                print("the daemon reached", n, flush=True)
    finally:
        print("the daemon's finally ran", flush=True)


threading.Thread(target=body, daemon=True).start()
time.sleep(0.3)
print("the main thread is done", flush=True)
"""


def run_child():
    """Start another copy of this interpreter and collect everything it printed."""
    if not sys.executable:
        return None
    try:
        done = subprocess.run(
            [sys.executable, "-c", CHILD],
            capture_output=True,
            text=True,
            timeout=60,
        )
    except OSError:
        return None
    return done.stdout.strip().splitlines()


said = run_child()
if said is None:
    print("  this build cannot start another process, so there is nothing to try here")
else:
    print("  the child printed", len(said), "lines, and the last one was:", said[-1])
    print("  did the daemon reach its finally block:", any("finally" in line for line in said))

> **Version note.** how far the daemon counts before the main thread returns depends entirely on the machine, so the number of lines moves from run to run, while the last line and the answer on the last line do not

That is a property of the thread state rather than of the lock, so it should look the same on a build that has no lock at all. The recording checks that on the free threaded image.

Does a daemon thread still get hung at shutdown on a build that has no lock to hang it with?

```python
"""A daemon thread that is still running when the interpreter shuts down.

Shutdown does not ask a daemon thread to stop. It stores one value into that thread's thread
state, and from then on the thread is allowed to keep running only until its next periodic
check. At that check it tries to attach, sees the value, and is hung where it stands. Its
finally blocks do not run, its with blocks do not exit, and nothing it was holding is released.

That is a property of the thread state rather than of the lock, so it should look exactly the
same on a build that has no lock at all. This program checks that, by starting a child that
counts in a daemon thread while its main thread sleeps briefly and then returns.
"""

import subprocess
import sys

CHILD = """
import threading
import time


def body():
    n = 0
    try:
        while True:
            n += 1
            if n % 1000000 == 0:
                print("the daemon reached", n, flush=True)
    finally:
        print("the daemon's finally ran", flush=True)


threading.Thread(target=body, daemon=True).start()
time.sleep(0.3)
print("the main thread is done", flush=True)
"""

print(f"the lock is on: {sys._is_gil_enabled()}")
done = subprocess.run(
    [sys.executable, "-c", CHILD],
    capture_output=True,
    text=True,
    check=True,
)
said = done.stdout.strip().splitlines()
ran_the_finally = any("finally ran" in line for line in said)
print(f"~ lines the child printed: {len(said)}")
print(f"the last line was: {said[-1]}")
print(f"the daemon reached its finally block: {ran_the_finally}")
print(f"the exit status was: {done.returncode}")
```

```text
the lock is on: False
~ lines the child printed: 6
the last line was: the main thread is done
the daemon reached its finally block: False
the exit status was: 0
```

That ran on Python 3.15.0rc1 in the freethreaded build this project publishes, which is `ghcr.io/tamnd/cpython-internals/cpython:freethreaded@sha256:db72284e3a49f43c38b96bec2baed1380b8348e27ea6f54f6e8d0810b59c3144`. You do not need that build to read the numbers, and you do need it to produce them, which is why this is recorded rather than left as a cell you run. If you want to watch it happen yourself, `docker run --rm -i ghcr.io/tamnd/cpython-internals/cpython:freethreaded@sha256:db72284e3a49f43c38b96bec2baed1380b8348e27ea6f54f6e8d0810b59c3144 python3 -` takes the program on standard input.

## Try it yourself

Four things, roughly in order of how much they will teach you.

Change the daemon thread in the last cell to `daemon=False` and run it again. The child takes a lot longer, prints far more, and its `finally` does run, because shutdown waits for it. That is the entire difference between the two kinds of thread.

Put a `with open(...)` around the daemon's loop instead of the `try`. The file is still open when the thread is hung, and the process exits anyway. It is worth seeing once, because it is the reason "the daemon will clean up after itself" is never true.

In the second cell, print `id(threading.current_thread())` alongside the other three numbers. Python object ids get recycled too, and for the same reason: the memory comes back around. Three of your four numbers are addresses wearing different hats.

Start a thread that does nothing but sleep for a long time, and walk the list from the main thread while it sleeps. Its state is still there. A sleeping thread is detached, not gone, which is the difference between a thread state and a running thread.

## What you now know

A thread, to CPython, is a struct. `PyThreadState` holds the frame it is running, the exception it is handling, its recursion budget, its scratch dict, the key its `threading.local` values hang off, and its place in a list.

That list hangs off the interpreter, newest first, and you can walk it from Python with five public C API calls and no debugger.

There are three numbers that name a thread. The state id is the interpreter's, counts up, and is never reused. `threading.get_ident()` and `get_native_id()` come from the operating system and both get handed out again after a thread exits.

`sys._current_frames()` and `sys._current_exceptions()` are that same walk with a different field read out, which is why they are keyed by ident.

The `state` field is where the GIL handshake actually happens. Attached and detached are the two the thread moves between itself, and `Py_BEGIN_ALLOW_THREADS` is a detach. Suspended is what stopping the world does to a thread. Shutting down is what finalization does, and a daemon thread that reads it is hung on the spot with none of its cleanup run.

## What is next

C04 goes up a level. The thread states in this lesson were all on one list hanging off the interpreter, and it turns out the interpreter is on a list too, hanging off the runtime. A process can hold several at once, each with its own GIL, its own modules and its own objects, which is a second way of using more than one core that has nothing to do with turning the lock off.